## Experiment 3 — Increase max_target_length to 512
**Hypothesis:** EDA showed 16.9% of training answers exceed 256 tokens, meaning they get truncated during training. Increasing max_target_length to 512 should capture more answer content and improve ROUGE scores.
**Changes from Experiment 1:**
- max_target_length: 256 → 512
- Keep epochs=1, lr=5e-4 (Experiment 1's best settings)
**Expected outcome:** Higher ROUGE/Zindi score due to less truncation, especially for longer-answer languages like Akan.

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [2]:
data_path = '/kaggle/input/datasets/oglorypaul/multilingual-health-qa/'

train = pd.read_csv(data_path + 'Train.csv')
val   = pd.read_csv(data_path + 'Val.csv')
test  = pd.read_csv(data_path + 'Test.csv')
sample_sub = pd.read_csv(data_path + 'SampleSubmission.csv')

print(f'Train: {train.shape}')
print(f'Val  : {val.shape}')
print(f'Test : {test.shape}')

Train: (29815, 4)
Val  : (6686, 4)
Test : (2618, 3)


In [3]:
MODEL_NAME = 'google/mt5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer loaded! Vocab size: 250,100


In [4]:
def preprocess(examples):
    inputs = tokenizer(
        examples['input'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        examples['output'],
        max_length=512,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

train_dataset = Dataset.from_pandas(train[['input', 'output']])
val_dataset   = Dataset.from_pandas(val[['input', 'output']])

train_tokenized = train_dataset.map(preprocess, batched=True)
val_tokenized   = val_dataset.map(preprocess, batched=True)

print(f'Train tokenized: {len(train_tokenized)}')
print(f'Val tokenized  : {len(val_tokenized)}')

Map:   0%|          | 0/29815 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Train tokenized: 29815
Val tokenized  : 6686


In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 968,342,784 || trainable%: 0.1827


In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/exp3_lora',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer.train()
print('Experiment 3 training complete!')


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,10.195087,2.060802


Experiment 3 training complete!


In [10]:
model.save_pretrained('/kaggle/working/exp3_final')
tokenizer.save_pretrained('/kaggle/working/exp3_final')
print('Exp3 model saved locally!')

Exp3 model saved locally!


## Experiment 4 — Increase LoRA Rank to 32
**Hypothesis:** A higher LoRA rank gives the model more trainable capacity, which may improve its ability to learn the mapping from health questions to long, detailed answers.
**Changes from Experiment 1:**
- LoRA rank: 16 → 32
- LoRA alpha: 32 → 64 (kept at 2x rank, standard practice)
- Keep epochs=1, lr=5e-4, max_target_length=256 (proven stable/fast settings)
**Expected outcome:** Possible score improvement from added capacity, at the cost of slightly more trainable parameters (still small relative to full model).

In [6]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [7]:
data_path = '/kaggle/input/datasets/oglorypaul/multilingual-health-qa/'

train = pd.read_csv(data_path + 'Train.csv')
val   = pd.read_csv(data_path + 'Val.csv')
test  = pd.read_csv(data_path + 'Test.csv')
sample_sub = pd.read_csv(data_path + 'SampleSubmission.csv')

print(f'Train: {train.shape}')
print(f'Val  : {val.shape}')
print(f'Test : {test.shape}')

Train: (29815, 4)
Val  : (6686, 4)
Test : (2618, 3)


In [8]:
MODEL_NAME = 'google/mt5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer loaded! Vocab size: 250,100


In [9]:
def preprocess(examples):
    inputs = tokenizer(
        examples['input'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        examples['output'],
        max_length=256,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

train_dataset = Dataset.from_pandas(train[['input', 'output']])
val_dataset   = Dataset.from_pandas(val[['input', 'output']])

train_tokenized = train_dataset.map(preprocess, batched=True)
val_tokenized   = val_dataset.map(preprocess, batched=True)

print(f'Train tokenized: {len(train_tokenized)}')
print(f'Val tokenized  : {len(val_tokenized)}')

Map:   0%|          | 0/29815 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Train tokenized: 29815
Val tokenized  : 6686


In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 3,538,944 || all params: 970,112,256 || trainable%: 0.3648


In [11]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/exp4_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer.train()
print('Experiment 4 training complete!')

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,7.389875,3.122357


Experiment 4 training complete!


In [12]:
model.save_pretrained('/kaggle/working/exp4_final')
tokenizer.save_pretrained('/kaggle/working/exp4_final')

import shutil
shutil.make_archive('/kaggle/working/exp4_final', 'zip', '/kaggle/working/exp4_final')
print('Zipped and ready to download!')

Zipped and ready to download!


In [13]:
model.eval()

predictions_exp4 = []
batch_size = 16

for i in tqdm(range(0, len(test), batch_size)):
    batch = test['input'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer(
        batch,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            no_repeat_ngram_size=3
        )
    
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions_exp4.extend(preds)

print(f'Generated {len(predictions_exp4)} predictions')
print(f'Sample: {predictions_exp4[0][:150]}')

100%|██████████| 164/164 [10:06<00:00,  3.70s/it]

Generated 2618 predictions
Sample: Eɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow ne akuo ahorow a ɛreyɛ adwumaso de asiw GBV ano ma nneyɛma sɛ wɔde wɔde adwumso de ayɛ GBV a w


In [14]:
import re

def clean_prediction(text):
    text = re.sub(r'^[^\x00-\x7F\u0100-\u024F\u1E00-\u1EFF]+', '', text)
    return text.strip()

predictions_exp4_clean = [clean_prediction(p) for p in predictions_exp4]

submission_exp4 = sample_sub.copy()
submission_exp4['TargetRLF1'] = predictions_exp4_clean
submission_exp4['TargetR1F1'] = predictions_exp4_clean
submission_exp4['TargetLLM']  = predictions_exp4_clean

submission_exp4.to_csv('/kaggle/working/submission_exp4.csv', index=False)
print('Submission saved!')
print(submission_exp4.head(3))

Submission saved!
                       ID                                         TargetRLF1  \
0  ID_TS_Aka_Gha_A3B1799D  Eɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛb...   
1  ID_TS_Aka_Gha_1C80317F  Ebɛtumi afi hokwan a mmabun wɔ sɛ wonya nipadu...   
2  ID_TS_Aka_Gha_06671AD1  Ewɔ nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ b...   

                                          TargetR1F1  \
0  Eɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛb...   
1  Ebɛtumi afi hokwan a mmabun wɔ sɛ wonya nipadu...   
2  Ewɔ nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ b...   

                                           TargetLLM  
0  Eɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛb...  
1  Ebɛtumi afi hokwan a mmabun wɔ sɛ wonya nipadu...  
2  Ewɔ nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ b...  


## Experiment 3 — Result Note
**Status:** Training completed (1 epoch, max_target_length=512) with Training Loss=10.20, Validation Loss=2.06 — both metrics suggested improved generalization compared to Experiment 1.
**Issue:** The Kaggle session crashed ("Draft Session Error") immediately after training completed, before the trained model could be saved or predictions generated. The session could not be recovered, and the working directory was wiped.
**Outcome:** No Zindi score available for this experiment. The validation loss improvement is recorded as evidence that increasing max_target_length is a promising direction, but it could not be confirmed via leaderboard submission due to infrastructure failure.
**Lesson learned:** Save and download trained models immediately after training completes, rather than waiting until after inference — implemented from Experiment 4 onward.

## Experiment 5 — Moderate Learning Rate at Rank=32
**Hypothesis:** Experiment 2 showed lr=1e-4 caused instability over multiple epochs, while Experiment 1/4 used lr=5e-4 successfully for 1 epoch. Testing a moderate lr=3e-4 at our best-known rank (32) may find a sweet spot for stability and performance.
**Changes from Experiment 4:**
- learning_rate: 5e-4 → 3e-4
- Keep r=32, alpha=64, epochs=1, max_target_length=256
**Expected outcome:** Potentially more stable training (lower loss) while maintaining or improving the 0.194928 Zindi score from Experiment 4.

In [18]:
del model
import gc
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 3,538,944 || all params: 970,112,256 || trainable%: 0.3648


In [19]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/exp5_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer.train()
print('Experiment 5 training complete!')

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,9.098635,3.692093


Experiment 5 training complete!


In [20]:
model.save_pretrained('/kaggle/working/exp5_final')
tokenizer.save_pretrained('/kaggle/working/exp5_final')

import shutil
shutil.make_archive('/kaggle/working/exp5_final', 'zip', '/kaggle/working/exp5_final')
print('Zipped and ready to download!')

Zipped and ready to download!


In [28]:
model.eval()

predictions_exp5 = []
batch_size = 16

for i in tqdm(range(0, len(test), batch_size)):
    batch = test['input'].iloc[i:i+batch_size].tolist()
    inputs = tokenizer(
        batch,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            no_repeat_ngram_size=3
        )
    
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions_exp5.extend(preds)

print(f'Generated {len(predictions_exp5)} predictions')

100%|██████████| 164/164 [09:42<00:00,  3.55s/it]

Generated 2618 predictions


In [29]:
import re

def clean_prediction(text):
    text = re.sub(r'^[^\x00-\x7F\u0100-\u024F\u1E00-\u1EFF]+', '', text)
    return text.strip()

predictions_exp5_clean = [clean_prediction(p) for p in predictions_exp5]

submission_exp5 = sample_sub.copy()
submission_exp5['TargetRLF1'] = predictions_exp5_clean
submission_exp5['TargetR1F1'] = predictions_exp5_clean
submission_exp5['TargetLLM']  = predictions_exp5_clean

nan_count = submission_exp5['TargetRLF1'].isna().sum()
print(f'NaN count: {nan_count}')
submission_exp5 = submission_exp5.fillna('No answer available')

submission_exp5.to_csv('/kaggle/working/submission_exp5.csv', index=False)
print('Submission saved!')

NaN count: 0
Submission saved!


In [31]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
kaggle_username = user_secrets.get_secret("KAGGLE_USERNAME")
kaggle_key = user_secrets.get_secret("KAGGLE_KEY")

import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('Kaggle API credentials configured!')

Kaggle API credentials configured!


In [32]:
!pip install kaggle -q

import os
os.makedirs('/kaggle/working/exp5_dataset', exist_ok=True)

# Copy the model files into the dataset folder
import shutil
shutil.copytree('/kaggle/working/exp5_final', '/kaggle/working/exp5_dataset/exp5_final', dirs_exist_ok=True)

# Create dataset metadata
metadata = {
    "title": "exp5-mt5-lora-model",
    "id": f"{kaggle_username}/exp5-mt5-lora-model",
    "licenses": [{"name": "CC0-1.0"}]
}

import json
with open('/kaggle/working/exp5_dataset/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

print('Dataset folder prepared!')

Dataset folder prepared!


In [33]:
!kaggle datasets create -p /kaggle/working/exp5_dataset

Skipping folder: exp5_final; use '--dir-mode' to upload folders
Dataset creation error: Please upload at least one file


In [34]:
!kaggle datasets create -p /kaggle/working/exp5_dataset --dir-mode zip

Starting upload for file exp5_final.zip
100%|██████████████████████████████████████| 16.1M/16.1M [00:00<00:00, 31.6MB/s]
Upload successful: exp5_final.zip (16MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/oglorypaul/exp5-mt5-lora-model


## Experiment 5 — Summary
**Setup:** mT5-base + LoRA (r=32), lr=3e-4, 1 epoch, max_target_length=256
**Result:** Training Loss=9.10, Validation Loss=3.69, Zindi Score=0.132925
**Insight:** Worse than Experiment 4 (lr=5e-4, score=0.194928), confirming lr=5e-4 is the better learning rate across all tested values (1e-4, 3e-4, 5e-4).
**Process note:** Kaggle's browser download and proxy links repeatedly failed for this model's files. Resolved by configuring the Kaggle API via notebook secrets and pushing the model as a permanent Kaggle Dataset (`exp5-mt5-lora-model`) — this is now the standard save method for all future experiments.

## Experiment 6 — Combine Best Rank with Longer Target Length
**Hypothesis:** Experiment 4 (r=32) improved over r=16, and Experiment 3 (max_target=512) showed better validation loss (2.06) than Experiment 1's 3.21, despite losing its Zindi score to a session crash. Combining both changes may compound the benefits.
**Changes from Experiment 4:**
- max_target_length: 256 → 512
- Keep r=32, alpha=64, epochs=1, lr=5e-4 (our best confirmed settings)
**Risk:** Longer sequences mean slower training and higher memory use — we'll need reduced batch size like we did for Experiment 3, and the run will take 1.5-3 hours.
**Mitigation:** Save to Kaggle Dataset via API immediately upon completion, before generating predictions, learning from Experiment 3's loss.

In [35]:
del model
import gc
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 3,538,944 || all params: 970,112,256 || trainable%: 0.3648


In [36]:
def preprocess(examples):
    inputs = tokenizer(
        examples['input'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        examples['output'],
        max_length=512,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

train_dataset = Dataset.from_pandas(train[['input', 'output']])
val_dataset   = Dataset.from_pandas(val[['input', 'output']])

train_tokenized = train_dataset.map(preprocess, batched=True)
val_tokenized   = val_dataset.map(preprocess, batched=True)

print(f'Train tokenized: {len(train_tokenized)}')
print(f'Val tokenized  : {len(val_tokenized)}')

Map:   0%|          | 0/29815 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Train tokenized: 29815
Val tokenized  : 6686


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/exp6_lora',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer.train()
print('Experiment 6 training complete!')

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


## Experiment 6 — Result & Process Note
**Status:** Training completed (1 epoch, r=32, max_target_length=512) with Training Loss=8.859055, Validation Loss=1.871400 — the best validation loss recorded across all experiments at this point.
**Issue:** Immediately after training completed, the Kaggle session encountered a 'Draft Session Error' combined with the GPU accelerator quota being exceeded (30-hour weekly limit reached). The kernel could not be reconnected, and the model weights were lost before they could be saved.
**Outcome:** No Zindi score available for this experiment, despite the strong validation loss result.
**Lesson learned:** This, combined with Experiment 3's similar failure, confirmed that max_target_length=512 runs (which take 2-3+ hours due to smaller batch sizes) carry significantly higher risk of session instability than the faster 256-length runs. This directly motivated saving models via the Kaggle API/dataset method (implemented from Experiment 5 onward) and switching to Google Drive-based saving on Colab for Experiment 7 onward, both of which proved more reliable.

## Experiment 7 — Task Prefix on Input
**Hypothesis:** T5-family models often perform better when given an explicit task instruction prefix, since they were pretrained on multi-task prefixed data. Adding "answer this health question: " before each input may help the model better understand the expected behavior.
**Changes from Experiment 4 (our best confirmed result):**
- Input format: raw question → "answer this health question: " + question
- Keep r=32, alpha=64, epochs=1, lr=5e-4, max_target_length=256 (all proven-best settings)
**Expected outcome:** Potential improvement in answer relevance/quality, testable at the same fast training speed (~50 min) as Experiment 1/4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers sentencepiece peft datasets rouge-score -q
print("Libraries installed!")

  Preparing metadata (setup.py) ... done
Libraries installed!


In [ ]:
!pip install -q peft --upgrade
!pip install -q torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.9 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [ ]:
data_path = '/content/drive/My Drive/multilingual-health-qa/'

train = pd.read_csv(data_path + 'Train.csv')
val   = pd.read_csv(data_path + 'Val.csv')
test  = pd.read_csv(data_path + 'Test.csv')
sample_sub = pd.read_csv(data_path + 'SampleSubmission.csv')

print(f'Train: {train.shape}')
print(f'Val  : {val.shape}')
print(f'Test : {test.shape}')

Train: (29815, 4)
Val  : (6686, 4)
Test : (2618, 3)


In [ ]:
MODEL_NAME = 'google/mt5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded! Vocab size: 250,100


In [ ]:
PREFIX = "answer this health question: "

def preprocess(examples):
    prefixed_inputs = [PREFIX + str(x) for x in examples['input']]

    inputs = tokenizer(
        prefixed_inputs,
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        examples['output'],
        max_length=256,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

train_dataset = Dataset.from_pandas(train[['input', 'output']])
val_dataset   = Dataset.from_pandas(val[['input', 'output']])

train_tokenized = train_dataset.map(preprocess, batched=True)
val_tokenized   = val_dataset.map(preprocess, batched=True)

print(f'Train tokenized: {len(train_tokenized)}')
print(f'Val tokenized  : {len(val_tokenized)}')

Map:   0%|          | 0/29815 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Train tokenized: 29815
Val tokenized  : 6686


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 3,538,944 || all params: 585,940,224 || trainable%: 0.6040


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/content/drive/My Drive/multilingual-health-qa/exp7_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer.train()
print('Experiment 7 training complete!')

Epoch,Training Loss,Validation Loss
1,3.524834,1.531013


Experiment 7 training complete!


In [ ]:
model.save_pretrained('/content/drive/My Drive/multilingual-health-qa/exp7_final')
tokenizer.save_pretrained('/content/drive/My Drive/multilingual-health-qa/exp7_final')
print('Exp7 model saved to Drive!')

Exp7 model saved to Drive!


In [ ]:
model.eval()

predictions_exp7 = []
batch_size = 16

for i in tqdm(range(0, len(test), batch_size)):
    batch = [PREFIX + str(x) for x in test['input'].iloc[i:i+batch_size].tolist()]
    inputs = tokenizer(
        batch,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            no_repeat_ngram_size=3
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions_exp7.extend(preds)

print(f'Generated {len(predictions_exp7)} predictions')
print(f'Sample: {predictions_exp7[0][:150]}')

100%|██████████| 164/164 [11:20<00:00,  4.15s/it]

Generated 2618 predictions
Sample: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma adwumayɛbea ahorow a wobɛyɛ adwumamayɛbeatia ahorow ne akuo ahorow, ne akuom ahorow ahorow de asiw GBV ano ma nn


In [ ]:
import re

def clean_prediction(text):
    text = re.sub(r'^[^\x00-\x7F\u0100-\u024F\u1E00-\u1EFF]+', '', text)
    return text.strip()

predictions_exp7_clean = [clean_prediction(p) for p in predictions_exp7]

submission_exp7 = sample_sub.copy()
submission_exp7['TargetRLF1'] = predictions_exp7_clean
submission_exp7['TargetR1F1'] = predictions_exp7_clean
submission_exp7['TargetLLM']  = predictions_exp7_clean

nan_count = submission_exp7['TargetRLF1'].isna().sum()
print(f'NaN count: {nan_count}')
submission_exp7 = submission_exp7.fillna('No answer available')

submission_exp7.to_csv('/content/drive/My Drive/multilingual-health-qa/submission_exp7.csv', index=False)
print('Submission saved to Drive!')

NaN count: 0
Submission saved to Drive!


In [ ]:
from google.colab import files
files.download('/content/drive/My Drive/multilingual-health-qa/submission_exp7.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Experiment 7 — Summary
**Setup:** mT5-base + LoRA (r=32), lr=5e-4, 1 epoch, max_target_length=256, task prefix "answer this health question: " added to all inputs
**Result:** Training Loss=3.52, Validation Loss=1.53 (best of all experiments so far), Zindi Score=0.195404 (new best)
**Insight:** Adding an explicit task prefix produced a dramatic drop in both loss metrics compared to Experiment 4 (same config, no prefix), but the Zindi score only improved marginally. This suggests the prefix significantly helped the model's confidence/fluency in generating plausible-looking text, but ROUGE-based scoring (which rewards exact word overlap with references) didn't capture the full extent of that improvement — a useful insight into the gap between training loss and competition metrics.

## Experiment 8 — Per-Language Performance Breakdown
**Hypothesis:** Given the dataset's language imbalance (English variants dominate, Amharic has the fewest examples), the model likely performs unevenly across languages. Breaking down ROUGE scores by language subset will reveal which languages are underserved by the current model.
**Method:** Using Experiment 7's trained model (current best), generate predictions on the validation set (which has ground-truth answers, unlike test) and compute ROUGE-1/ROUGE-L scores separately for each of the 8 language subsets.
**No new training required** — this evaluates the existing best model from a different analytical angle.
**Expected outcome:** Likely lower scores for Amharic (least training data) and possibly Akan (longest, most complex answers), informing future improvement directions.

In [ ]:
model.eval()

# Generate predictions on the validation set (has ground-truth answers for comparison)
val_predictions = []
batch_size = 16

for i in tqdm(range(0, len(val), batch_size)):
    batch = [PREFIX + str(x) for x in val['input'].iloc[i:i+batch_size].tolist()]
    inputs = tokenizer(
        batch,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            no_repeat_ngram_size=3
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    val_predictions.extend(preds)

print(f'Generated {len(val_predictions)} validation predictions')

100%|██████████| 418/418 [27:28<00:00,  3.94s/it]

Generated 6686 validation predictions


In [16]:
from rouge_score import rouge_scorer

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
    rouge1_scores, rougeL_scores = [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(str(ref), str(pred))
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    return {
        'rouge1': sum(rouge1_scores) / len(rouge1_scores),
        'rougeL': sum(rougeL_scores) / len(rougeL_scores)
    }

results_by_language = {}
val_with_preds = val.copy()
val_with_preds['prediction'] = val_predictions

for lang in val_with_preds['subset'].unique():
    lang_df = val_with_preds[val_with_preds['subset'] == lang]
    scores = compute_rouge(lang_df['prediction'].tolist(), lang_df['output'].tolist())
    results_by_language[lang] = scores
    print(f"{lang:12s} | n={len(lang_df):5d} | ROUGE-1: {scores['rouge1']:.4f} | ROUGE-L: {scores['rougeL']:.4f}")

Aka_Gha      | n= 1114 | ROUGE-1: 0.3359 | ROUGE-L: 0.2342
Amh_Eth      | n=  462 | ROUGE-1: 0.0110 | ROUGE-L: 0.0110
Eng_Eth      | n=  564 | ROUGE-1: 0.2253 | ROUGE-L: 0.1853
Eng_Gha      | n= 1104 | ROUGE-1: 0.2947 | ROUGE-L: 0.2317
Eng_Ken      | n=  390 | ROUGE-1: 0.1729 | ROUGE-L: 0.1272
Eng_Uga      | n= 1688 | ROUGE-1: 0.1617 | ROUGE-L: 0.1167
Lug_Uga      | n=  846 | ROUGE-1: 0.1278 | ROUGE-L: 0.1058
Swa_Ken      | n=  518 | ROUGE-1: 0.1914 | ROUGE-L: 0.1485


## Experiment 8 — Results & Analysis
| Language | n | ROUGE-1 | ROUGE-L |
|---|---|---|---|
| Aka_Gha | 1114 | 0.336 | 0.234 |
| Eng_Gha | 1104 | 0.295 | 0.232 |
| Eng_Eth | 564 | 0.225 | 0.185 |
| Swa_Ken | 518 | 0.191 | 0.149 |
| Eng_Ken | 390 | 0.173 | 0.127 |
| Eng_Uga | 1688 | 0.162 | 0.117 |
| Lug_Uga | 846 | 0.128 | 0.106 |
| **Amh_Eth** | 462 | **0.011** | **0.011** |

**Key finding:** Amharic performance is catastrophically lower than every other language, despite having a similar example count to Swa_Ken and Eng_Ken which performed reasonably well. This rules out "data quantity" as the sole explanation.

**Hypothesis:** Amharic is the only language in this dataset written in Ge'ez script (Ethiopic), unlike all others which use Latin-based scripts. mT5's tokenizer and pretraining may handle this script far less effectively than Latin-script African languages like Akan and Luganda.

**Insight for Akan's strong performance:** Despite being low-resource, Akan (Latin script with diacritics) outperformed even some English variants — suggesting script familiarity matters more than raw resource availability for this model.

In [17]:
amharic_examples = val_with_preds[val_with_preds['subset'] == 'Amh_Eth'].head(3)
for idx, row in amharic_examples.iterrows():
    print('QUESTION :', row['input'][:100])
    print('REFERENCE:', row['output'][:100])
    print('PREDICTED:', row['prediction'][:100])
    print('---')

QUESTION : ለእናቴ የ አባላዘር በሽታ ሊይዘኝ እንደሚችል መንገር እፈራለሁ። ሌላ ማንን ማነጋገር እችላለሁ?
REFERENCE: ከማህበረሰብ የጤና ሰራተኛ፣ ከክሊኒክ ነርስ ወይም ከታመነች እህት ወይም ሴት ዘመድ ጋር ይነጋገሩ። የጤና መረጃዎን በሚስጥር መያዝ ይጠበቅባቸዋል።
PREDICTED: አባላዘር በሽታ ሊይዘኝ አይችልም። አባላት በሽታው ሊይዝ ይችላል።
---
QUESTION : በተደጋጋሚ በሚከሰት የሄርፒስ ኢንፌክሽን ምክንያት የተለመደው የቁስል ገጽታ ምን ይመስላል?
REFERENCE: ብዙውን ጊዜ በመጠን እና በቁጥር ያነሱ ናቸው፤ በፍጥነት ይድናሉ፤ እናም ከመጀመሪያውው ኢንፌክሽን ይልቅ በትንሽ ቦታ የተወሰኑ ናቸው።
PREDICTED: ግን ግን የቁስል ገጽታ ምን ይመስላል? ግን በዚህ ጊዜ ግን ይህ ምክንያት የሄርፒስ ኢንፌክሽን ምክያት ምልክቶች ላይ መድሃኒት ካለበት ምክር ካለው ትክክቶች ም
---
QUESTION : አስፈሪ የምርመራ ውጤት ሳገኝ ጠንካራ እንድሆን ለመርዳት ሃይማኖታዊ እምነቴን መጠቀም እችላለሁ?
REFERENCE: አዎለሰላም ለመጸለይ እምነትህን ተጠቀም፣ አእምሮህን ለማረጋጋት እና ማንኛውንም ነገር ለመጋፈጥ ጥንካሬ እንዳለህ አስታውስ።
PREDICTED: ምርመራ ውጤት ካገኝ ጠንካራ እንድሆን ለመርዳት ጥሩ ውጣት ያገኝ።
---


## Experiment 8 — Qualitative Verification
Inspecting actual Amharic predictions confirms the script renders correctly (no encoding/tokenization corruption) and the model produces grammatically plausible Amharic text. However, predictions are generic, repetitive, or merely restate the question rather than conveying the specific medical content present in the reference answer. This suggests the failure is not a script-handling issue but rather insufficient learned content-mapping for Amharic — likely because mT5's multilingual pretraining included substantially less Amharic text than Latin-script African languages, even though our fine-tuning set size was comparable across languages.

## Experiment 9 — Inference Strategy Comparison
**Hypothesis:** Different generation/decoding strategies at inference time may improve answer quality without any retraining. Comparing greedy decoding, beam search, and varying repetition penalties on Experiment 7's existing trained model.
**Method:** Using the same Experiment 7 model, generate predictions on a validation sample (500 examples for speed) under three settings:
1. Greedy decoding (current default used in Exp 1-8)
2. Beam search (num_beams=4)
3. Beam search + higher no_repeat_ngram_size

Compare ROUGE-1/ROUGE-L across all three to identify the best inference configuration.
**No new training required** — this tests inference strategy, a distinct experiment type explicitly listed in the rubric.

In [18]:
val_sample = val.sample(500, random_state=42).reset_index(drop=True)

def generate_with_settings(questions, **gen_kwargs):
    preds = []
    batch_size = 16
    for i in range(0, len(questions), batch_size):
        batch = [PREFIX + str(x) for x in questions[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors='pt', max_length=128, truncation=True, padding=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128, **gen_kwargs)
        preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

print('Running greedy...')
preds_greedy = generate_with_settings(val_sample['input'].tolist(), no_repeat_ngram_size=3)

print('Running beam search...')
preds_beam = generate_with_settings(val_sample['input'].tolist(), num_beams=4, no_repeat_ngram_size=3, early_stopping=True)

print('Done!')

Running greedy...
Running beam search...
Done!


In [19]:
references = val_sample['output'].tolist()

scores_greedy = compute_rouge(preds_greedy, references)
scores_beam = compute_rouge(preds_beam, references)

print(f"Greedy decoding    | ROUGE-1: {scores_greedy['rouge1']:.4f} | ROUGE-L: {scores_greedy['rougeL']:.4f}")
print(f"Beam search (n=4)  | ROUGE-1: {scores_beam['rouge1']:.4f} | ROUGE-L: {scores_beam['rougeL']:.4f}")

Greedy decoding    | ROUGE-1: 0.1939 | ROUGE-L: 0.1469
Beam search (n=4)  | ROUGE-1: 0.1848 | ROUGE-L: 0.1452


## Experiment 9 — Results & Analysis
| Decoding Strategy | ROUGE-1 | ROUGE-L |
|---|---|---|
| Greedy (no_repeat_ngram=3) | 0.1939 | 0.1469 |
| Beam search (num_beams=4) | 0.1848 | 0.1452 |

**Finding:** Greedy decoding slightly outperformed beam search on this validation sample, contrary to common assumption that beam search improves generation quality. A likely explanation: beam search optimizes for overall sequence probability/fluency, which can favor safer, more generic phrasing — while ROUGE specifically rewards exact word-overlap with references, which greedy's more "direct" token-by-token choices may align with better in this fine-tuned, narrow-domain setting.
**Decision:** Greedy decoding remains the inference method used for all submissions, validated as the better choice rather than assumed by default.

## Experiments 10–14 — Retrieval-Based Approaches (Run Locally, No GPU)

After completing fine-tuning experiments 1–9, the best confirmed Zindi score was 0.195404 (Experiment 7). To explore a fundamentally different methodology, retrieval-based approaches were tested locally on CPU, motivated by the insight that ROUGE metrics reward exact word/phrase overlap with reference answers — a property that retrieval directly exploits by surfacing real training answers rather than generating new text.

**Experiment 10 — Basic Retrieval:** For each test question, encoded all questions using a multilingual sentence embedding model (`paraphrase-multilingual-MiniLM-L12-v2`) and retrieved the single most similar training question's answer via cosine similarity, regardless of language. **Score: 0.479688** — more than double the best fine-tuning result.

**Experiment 11 — Language-Aware Retrieval:** Restricted retrieval to only match within the same language subset (e.g., Amharic questions only retrieve from Amharic training answers), preventing cross-language mismatches. **Score: 0.486525.**

**Experiment 12 — Top-3 Concatenation:** Instead of the single best match, concatenated the top 3 most similar answers per question, hypothesizing more text could increase ROUGE overlap. This underperformed (**Score: 0.411156**), likely because concatenation introduced repetition and irrelevant content that reduced precision.

**Experiment 13 — Stronger Embedding Model (mpnet):** Replaced MiniLM with the larger `paraphrase-multilingual-mpnet-base-v2` model, keeping language-aware single-best-match retrieval. This produced the best result so far: **Score: 0.510140.**

**Experiment 14 — Question+Answer Combined Embedding:** Tested encoding training examples using both question and answer text combined, hypothesizing richer representations might improve matching. This underperformed (**Score: 0.432645**), since test questions only contain question text, making the comparison basis inconsistent.

**Best retrieval configuration carried forward:** language-aware, single best match, mpnet embeddings (Experiment 13, 0.510140) — used as the retrieval component for Experiment 15 below.


## Experiment 15 — Retrieval-Augmented Generation (RAG)
Hypothesis: Combining retrieval (best: 0.510140) with the fine-tuned model (best: 0.195404) may outperform either alone, by letting the model refine retrieved reference answers rather than copying them directly or generating purely from memory. Method: Retrieve the most similar training answer per test question (Experiment 13's approach), then feed it as context into the Experiment 7 fine-tuned model via the prompt: "answer this health question using the reference: [retrieved answer] question: [test question]". Expected outcome: Potential score improvement beyond 0.510140 if the model successfully grounds generation in retrieved context.

In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!pip install transformers sentencepiece peft datasets sentence-transformers scikit-learn -q
!pip install -q peft --upgrade
!pip install -q torchao --upgrade
print('Libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.4 MB/s eta 0:00:00
Libraries installed!


In [8]:
import torch
import pandas as pd
import numpy as np
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
data_path = '/content/drive/My Drive/multilingual-health-qa/'
train = pd.read_csv(data_path + 'Train.csv')
test  = pd.read_csv(data_path + 'Test.csv')
sample_sub = pd.read_csv(data_path + 'SampleSubmission.csv')
print(f'Train: {train.shape}, Test: {test.shape}')

Train: (29815, 4), Test: (2618, 3)


In [11]:
MODEL_NAME = 'google/mt5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = PeftModel.from_pretrained(base_model, data_path + 'exp7_final')
model = model.to(device)
model.eval()
print(next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

cuda:0


In [12]:
print("Loading embedding model...")
embed_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=device)

test['subset'] = test['ID'].apply(lambda x: '_'.join(x.split('_')[2:4]))

print("Retrieving best matching answers...")
retrieved_answers = {}

for lang in test['subset'].unique():
    print(f"Processing: {lang}")
    test_lang  = test[test['subset'] == lang]
    train_lang = train[train['subset'] == lang]
    if len(train_lang) == 0:
        train_lang = train

    test_emb  = embed_model.encode(test_lang['input'].tolist(), batch_size=64, show_progress_bar=False)
    train_emb = embed_model.encode(train_lang['input'].tolist(), batch_size=64, show_progress_bar=False)

    sims = cosine_similarity(test_emb, train_emb)
    best_idx = np.argmax(sims, axis=1)

    for i, idx in enumerate(best_idx):
        retrieved_answers[test_lang['ID'].iloc[i]] = train_lang['output'].iloc[idx]

print(f'Retrieved {len(retrieved_answers)} reference answers')

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieving best matching answers...
Processing: Aka_Gha
Processing: Amh_Eth
Processing: Eng_Eth
Processing: Eng_Gha
Processing: Eng_Ken
Processing: Eng_Uga
Processing: Lug_Uga
Processing: Swa_Ken
Retrieved 2618 reference answers


In [13]:
def truncate_text(text, max_chars=300):
    return str(text)[:max_chars]

predictions_exp15 = []
batch_size = 8

test_ids = test['ID'].tolist()
test_questions = test['input'].tolist()

for i in tqdm(range(0, len(test), batch_size)):
    batch_ids = test_ids[i:i+batch_size]
    batch_questions = test_questions[i:i+batch_size]

    prompts = []
    for tid, q in zip(batch_ids, batch_questions):
        ref = truncate_text(retrieved_answers[tid])
        prompt = f"answer this health question using the reference: {ref} question: {q}"
        prompts.append(prompt)

    inputs = tokenizer(prompts, return_tensors='pt', max_length=256, truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, no_repeat_ngram_size=3)

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions_exp15.extend(preds)

print(f'Generated {len(predictions_exp15)} RAG predictions')
print(f'Sample: {predictions_exp15[0][:150]}')

100%|██████████| 328/328 [21:39<00:00,  3.96s/it]

Generated 2618 RAG predictions
Sample: Nsɛm a edi mu a ɛfa nyinsɛn ano a wosiw ho ma wo ho tumi na ama woasisi gyinae a wɔfata wɔ wo nna mu akwahosan ho adwumayɛfo ne ahyehyɛde ahorow a won


In [14]:
def clean_prediction(text):
    text = re.sub(r'^[^\x00-\x7F\u0100-\u024F\u1E00-\u1EFF]+', '', str(text))
    return text.strip()

predictions_exp15_clean = [clean_prediction(p) for p in predictions_exp15]

submission_exp15 = sample_sub.copy()
submission_exp15['TargetRLF1'] = predictions_exp15_clean
submission_exp15['TargetR1F1'] = predictions_exp15_clean
submission_exp15['TargetLLM']  = predictions_exp15_clean

nan_count = submission_exp15['TargetRLF1'].isna().sum()
print(f'NaN count: {nan_count}')
submission_exp15 = submission_exp15.fillna('No answer available')

submission_exp15.to_csv('/content/drive/My Drive/multilingual-health-qa/submission_exp15.csv', index=False)
print('Saved to Drive!')

NaN count: 0
Saved to Drive!


In [15]:
from google.colab import files
files.download('/content/drive/My Drive/multilingual-health-qa/submission_exp15.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Experiment 15 — Retrieval-Augmented Generation (RAG)
**Hypothesis:** Combining retrieval (0.510) with fine-tuned generation (0.195) may outperform either alone.
**Method:** Fed the retrieved answer as context into the fine-tuned model: "answer using reference: [answer] question: [question]"
**Result:** Score = 0.228704
**Insight:** RAG beat pure fine-tuning but underperformed pure retrieval — the model rewrites text even with correct context, reducing exact word-overlap that ROUGE rewards. Preserving retrieved text verbatim beats refining it.

## Experiment 16 — BM25 Keyword Retrieval
**Hypothesis:** BM25's exact keyword matching may align better with ROUGE than semantic embeddings.
**Method:** Replaced mpnet embeddings with BM25 keyword scoring, same language-aware setup.
**Result:** Score = 0.461861
**Insight:** Underperformed semantic embeddings (0.510) — paraphrased questions need meaning-based matching, not just keyword overlap.

## Final Best Approach
**Experiment 13** — language-aware retrieval, mpnet embeddings, single best match — **Score: 0.510140**

## Experiment 16 — BM25 Keyword Retrieval
**Hypothesis:** BM25's exact keyword matching may align better with ROUGE than semantic embeddings.
**Method:** Replaced mpnet embeddings with BM25 keyword scoring, same language-aware setup. Implemented in `src/retrieval_bm25.py`, run locally (no GPU required).
**Result:** Score = 0.461861
**Insight:** Underperformed semantic embeddings (0.510) — paraphrased questions need meaning-based matching, not just keyword overlap.

In [ ]:
# Run via terminal: python3 src/retrieval_bm25.py
# See src/retrieval_bm25.py for full implementation.
# Output: outputs/submissions/submission_bm25.csv -> Zindi score 0.461861

## Final Best Approach
**Experiment 13** — language-aware retrieval, mpnet embeddings, single best match — **Score: 0.510140**